# MediFlow 1차 모델 비교 · 6개 학습

공통 ZIP `MediFlow_Adaptation_Common_v1.zip`을 **내 드라이브 → mediflow_Project** 바로 아래에 한 번 올립니다.
이 노트북에서 **GPU 선택 → 모두 실행 → Drive 연결**을 합니다.

피부·얼굴·두피의 기본/선정 방식에 같은 조명 변화를 적용합니다. 출발 모델을 새로 준비해야
하는 조건에는 준비 epoch가 표시됩니다. `Epoch 1/3`, 배치 수, loss, 검증 점수와 저장 단계가
실시간으로 보입니다. 재실행은 완료한 epoch까지 복원합니다. 동시에 여러 세션에서 실행하지 않습니다.

결과는 내 드라이브의 `MediFlow_adaptation_runs/e2b93d8899579c6b/stage1`에 저장합니다.
1차 선택은 원본·조명 변화 Validation 결과로 결정합니다.
2·3차에서 사용할 선택 모델의 1차 이전 가중치도 `initial.keras`로 별도 저장합니다.

공통 ZIP의 실제 촬영 입력은 암호화되어 있습니다. 1차는 복호화 키 없이 실행합니다.
3차에서 사용할 별도 키 파일은 로컬에 보관하세요.


In [ ]:
import hashlib
import json
import shutil
import stat
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath


def digest(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            value.update(chunk)
    return value.hexdigest()


def unpack_bundle(bundle, destination, expected_hash):
    """Verify the previously supplied bundle before using any of its Python files."""
    if digest(bundle) != expected_hash:
        raise ValueError(
            "선택한 ZIP이 준비된 실행 ZIP과 다릅니다. MediFlow_Adaptation_Common_v1.zip을 선택하세요."
        )
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(bundle) as archive:
        manifest = json.loads(archive.read("bundle_manifest.json"))
        members = {}
        for info in archive.infolist():
            name = info.orig_filename
            parts = PurePosixPath(name).parts
            if (
                not parts
                or PurePosixPath(name).is_absolute()
                or ".." in parts
                or "\\" in name
                or ":" in name
                or stat.S_ISLNK(info.external_attr >> 16)
                or name.casefold() in members
            ):
                raise ValueError("안전하지 않거나 중복된 ZIP 경로: " + name)
            members[name.casefold()] = info
        files = {info.filename for info in members.values() if not info.is_dir()}
        if files != set(manifest) | {"bundle_manifest.json"}:
            raise ValueError("실행 ZIP의 파일 목록이 검증 목록과 다릅니다.")
        for name, expected in manifest.items():
            body = archive.read(name)
            if hashlib.sha256(body).hexdigest() != expected:
                raise ValueError("실행 파일 검사 실패: " + name)
            path = destination / name
            if not path.resolve().is_relative_to(destination):
                raise ValueError("ZIP 경로가 실행 폴더 밖을 가리킵니다.")
            path.parent.mkdir(parents=True, exist_ok=True)
            with path.open("xb") as stream:
                stream.write(body)
    return destination

from google.colab import drive

drive.mount("/content/drive")
MY_DRIVE = Path("/content/drive/MyDrive")
PROJECT = MY_DRIVE / "mediflow_Project"
BUNDLE = PROJECT / "MediFlow_Adaptation_Common_v1.zip"
if not BUNDLE.is_file():
    raise FileNotFoundError(f"공통 ZIP을 {PROJECT} 바로 아래에 올리세요: {BUNDLE.name}")
print("Drive 공통 ZIP 확인 중:", BUNDLE, flush=True)
EXPECTED = "e2b93d8899579c6bf33878121a394c2572c70b91b5015252f2e88f9f8c89ff11"
with tempfile.TemporaryDirectory(prefix="mediflow_stage1_", dir="/content") as temporary:
    work = Path(temporary)
    local_bundle = work / BUNDLE.name
    shutil.copyfile(BUNDLE, local_bundle)
    runtime = unpack_bundle(local_bundle, work / "runtime", EXPECTED)
    sys.path.insert(0, str(runtime / "src"))
    try:
        from mediflow_datasets.adaptation_stage1_colab import launch
        STAGE1_RESULTS = launch(runtime, work, PROJECT, EXPECTED, MY_DRIVE)
    finally:
        sys.path.remove(str(runtime / "src"))
